# Part 1
Understanding the dataset

1.1. [1 point] Why are you interested in this dataset?

As students, we found it interesting to be able to analyze the impacts of studying and reviewing habits to see how those factors can be modeled to predict a student’s level of understanding. Using this data set for this assignment will not only help us understand the concepts of PCA, k-mean and DBSCAN, but also give us good insights to how study habits may impact our acidemic performance.

1.2. [1 point] How many numerical attributes and categorical attributes are there in the dataset?

Numerical attributes: 5 (STG, SCG, STR, LPR, PEG)
Categorical attributes: 1 (UNS)

1.3. [2 point] Are there any missing values in numerical attributes? If so, how are you going to
handle them? (Are you going to remove all data instances with missing values?) Will all
attributes with missing values be removed? Will missing values be imputed? If so, how?

There are no missing values.

1.4. Before doing any analysis, answer the following questions.
1.4.1. [1 point] Why do you think clusters will be present in this dataset?


Clusters will be present because students tend to have similar patterns in their study habits and performance. We expect that students with similar combinations of studying, practice, and exam scores will group together, forming natural clusters in the data.

1.4.2. [1 point] Why might finding clusters in this dataset be helpful? (How might this help us to
understand or analyze data?)


Finding clusters is helpful because it groups students with similar study behaviors and performance together. For example, it can show groups of students who consistently perform well with strong study habits, and students who struggle despite studying. This helps break the data into meaningful categories that are easier to analyze and interpret.

1.4.3. [1 point] How many clusters do you expect to see in this dataset? You can provide a range
of values for this question.   (Ex: 2-5). Why do you expect to see this number of clusters in
this dataset?


About 3–4 clusters.

Low performing / Low study students
Medium performing students with average study habits
High performing / High study students
Possibly a mixed or borderline group where study habits and performance don’t clearly match

1.4.4. [2 point] Do you expect to see clusters of similar size? Why or why not?

We do not expect clusters to be the same size because the data is not evenly distributed. Some types of students (like average study habits and average scores) are likely to appear more often than others, so those groups will form larger clusters, while less common patterns will form smaller ones.

# Part 2
Write the following functions in Python. You may use the scikit-learn or other packages to
check the correctness of your implementation, but you may not use existing clustering algorithm
implementations in your code. Provide test cases for your functions. You can use existing data
structures and basic helper functions numpy, pandas, etc

In [3]:
import numpy as np

2.1. [10 points] A function that implements k-means clustering algorithm. This function should take
a data matrix, a number of clusters k, and a convergence parameter 𝝐 (𝒆𝒑𝒔𝒊𝒍𝒐𝒏), as inputs,
and return the representatives (means) as well as the clusters found using k-means. If the
distance is the same between a point and more than one representative (mean), then assign
the point to the mean corresponding to the cluster with the lowest index.


In [4]:
def kmeans(X, k, epsilon):
    indices = np.random.choice(len(X), k, replace=False)
    centroids = X[indices]

    while True:
        # assign each point to nearest centroid
        labels = np.zeros(len(X), dtype=int)
        for i, point in enumerate(X):
            distances = [np.linalg.norm(point - centroid) for centroid in centroids]
            labels[i] = np.argmin(distances)

        # recompute centroids
        new_centroids = np.zeros_like(centroids)
        for i in range(k):
            points_in_cluster = X[labels == i]
            if len(points_in_cluster) > 0:
                new_centroids[i] = np.mean(points_in_cluster, axis=0)
            else:
                new_centroids[i] = centroids[i]

        # check convergence
        if np.max(np.abs(new_centroids - centroids)) < epsilon:
            break

        centroids = new_centroids

    return centroids, labels


# Two separated clusters
X_test = np.array([
    [1, 1], [1.5, 1], [1, 1.5],   # cluster A
    [9, 9], [9.5, 9], [9, 9.5]    # cluster B
])

centroids, labels = kmeans(X_test, k=2, epsilon=0.0001)
print("Labels:", labels)   # should be [0,0,0,1,1,1] or [1,1,1,0,0,0]
print("Centroids:", centroids)

Labels: [1 1 1 0 0 0]
Centroids: [[9.16666667 9.16666667]
 [1.16666667 1.16666667]]


2.2. [10 points] A function that implements DBSCAN clustering algorithm. The function should take
a data matrix and the parameters, minpts and 𝝐 (epsilon) as inputs, and return the clusters
found using DBSCAN, where each data point is labeled as either a noise point, a border point,
or a core point.


In [5]:
def dbscan(X, minpts, epsilon):
    labels = np.full(len(X), -1)
    point_types = np.full(len(X), "noise", dtype=object)
    cluster_id = 0

    visited = set()
    for i in range(len(X)):
        if i in visited:
            continue
        else:
            visited.add(i)

        # find all neighbors within epsilon distance
        neighbors = get_neighbors(X, i, epsilon)
        if len(neighbors) < minpts:
            # not enough neighbors, mark as noise
            labels[i] = -1
        else:
            # enough neighbors, start a new cluster and expand it
            expand_cluster(X, labels, point_types, neighbors, i, cluster_id, minpts, epsilon, visited)
            cluster_id += 1


    return labels, point_types

def get_neighbors(X, point_id, epsilon):
    # collect all points within epsilon distance of point_id
    neighbors = []
    for i, point in enumerate(X):
        if np.linalg.norm(X[point_id] - point) < epsilon:
            neighbors.append(i)
    return neighbors

def expand_cluster(X, labels, point_types, neighbors, i, cluster_id, minpts, epsilon, visited):
    # assign core point to cluster
    labels[i] = cluster_id
    point_types[i] = "core"

    queue = list(neighbors)

    while queue:
        j = queue.pop(0)

        if j not in visited:
            visited.add(j)
            # check if j is also a core point
            j_neighbors = get_neighbors(X, j, epsilon)

            if len(j_neighbors) >= minpts:
                # j is a core point, expand cluster through its neighbors
                point_types[j] = "core"
                queue += [n for n in j_neighbors if n not in visited]
            else:
                # j is reachable but can't expand further
                point_types[j] = "border"

        # absorb point into current cluster if unassigned
        if labels[j] == -1:
            labels[j] = cluster_id

X_test = np.array([
    [1, 1], [1.2, 1], [1.1, 1.2],   # cluster A
    [8, 8], [8.2, 8], [8.1, 8.2],   # cluster B
    [50, 50]                          # noise
])

labels, point_types = dbscan(X_test, minpts=2, epsilon=0.5)
print("Labels:", labels)      # should be [0,0,0,1,1,1,-1]
print("Types:", point_types)  # cluster points = core/border, last = noise

Labels: [ 0  0  0  1  1  1 -1]
Types: ['core' 'core' 'core' 'core' 'core' 'core' 'noise']


2.3. [3 points] Extra Credit: A function that computes the precision of clustering. The function
should take a list of true cluster labels and a list of predicted cluster labels returned by a
clustering algorithm and return the precision of the clustering.

# Part 3
 Analyze your data. Report the following, using tables or figures as appropriate. You may use
scikit-learn’s implementation of k-means and DBSCAN, but you are encouraged to first try using your
own implementations on real-world data

3.1. [5 points] First use sklearn’s PCA library to linearly transform the dataset into two dimensions.
Create a scatter plot of the dataset where x-axis corresponds to 1st principal component and yaxis corresponds to 2nd principal component. Does it look like there are clusters in the dataset
in these two dimensions? If so, how many clusters are there?

3.2. [4 points] Use the sklearn’s PCA library without inputting the number of components to use.
Then create a plot where x-axis is the number of principal components and y-axis is the
fraction of variance captured by the first r principal components. Based on this plot, choose a
number of principal components to reduce the dimensionality of the data. Write down the
fraction of the variance captured by the chosen set of principal components.


3.3. [6 points] For both original and reduced dimensionality data obtained in 3.2 do the following.
Run the k-means algorithm on a range of values for number of clusters k. Use at least 6
different k values for this. Plot the objective function value for each of the k values that you
used. Note that since you are doing this for original and dimensionality reduced datasets, you
need to create two plots. For original and dimensionality reduced datasets, what would be
good k-values based on the plots?


3.4. [6 points] For both original and reduced dimensionality data obtained in 3.2 do the following.
Run the DBSCAN algorithm with a range of values for the minpts and epsilon parameters to
find clusters in the datasets. First, you can keep the minpts fixed and vary the epsilon parameter, then you can keep the epsilon parameter fixed and vary the minpts parameter. Use
at least 6 different values for minpts and 6 different values for epsilon. Report the number of
clusters found for each minpts and epsilon pair.

3.5. [3 points] Extra credit: Create a plot of clustering precision for each value of k used in question
3.3, then create a plot of clustering precision for each value of epsilon used in question 3.4,
then create a plot of clustering precision for each value of minpts used in question 3.4. You
need to do this for both original and reduced dimensionality datasets. Overall, there should be
6 plots in total for this.